# Lecture 28 Worksheet: Many-to-Many RNNs
**NAVTTC / NIAI AI/ML Trainer Cohort** — works in Google Colab. Fill every `___` blank and run top to bottom; check lines print `OK`. Tasks 1 and 2 use numpy only. Tasks 3 and 4 train small networks with TensorFlow, which Colab already has (about a minute each on a CPU).

## Task 1: Which type of RNN?
Use the three questions from slide 3. For each task, fill in one of: `"many-to-one"`, `"same length"`, `"same length, bidirectional"`, `"length changes"`.

In [ ]:
types = {
    "Is this review positive?":                              "many-to-one",
    "Mark every name and date in a sentence":               "same length",
    "Mark each word, where the word AFTER it matters too":  "same length, bidirectional",
    "Turn a long email into a 2-line summary":              "length changes",
}
answers = ["many-to-one", "same length", "same length, bidirectional", "length changes"]
for q, a in types.items(): print(f"{a:28s} <- {q}")
assert list(types.values()) == answers; print("OK")

## Task 2: An output at every step
Same cell as Lecture 27. This time keep the output at every step: `y = W_hy * h + b_y`. Compare with slide 6 (1.42, 2.16, 2.40).

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
W_xh, W_hh, b_h = 0.5, 0.4, 0.0
W_hy, b_y = 2.0, 0.5

def rnn_outputs(x):
    h, hs, ys = 0.0, [], []
    for x_t in x:
        h = np.tanh(W_hh * h + W_xh * x_t + b_h)
        hs.append(h)
        ys.append(W_hy * h + b_y)      # read the memory at EVERY step
    return np.array(hs), np.array(ys)

hs, ys = rnn_outputs([1, 2, 3])
print("memories:", hs, " outputs:", ys)
assert np.allclose(ys, [1.42, 2.16, 2.40], atol=0.01); print("OK")

import tensorflow as tf
L = tf.keras.layers
T = 20

def make_cum(N, seed):
    r = np.random.RandomState(seed)
    x = r.randn(N, T, 1).astype("float32")
    y = (np.cumsum(x[:, :, 0], axis=1) > 0).astype("float32")
    return x, y

xtr, ytr = make_cum(3000, 0); xte, yte = make_cum(1000, 1)

def fit_eval(layers, xtr, ytr, xte, yte, seed=0):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential(layers)
    m.compile(tf.keras.optimizers.Adam(0.01), "binary_crossentropy")
    m.fit(xtr, ytr, epochs=15, batch_size=64, verbose=0)
    p = (m.predict(xte, verbose=0)[..., 0] > 0.5)
    return p, m

p0, _ = fit_eval([L.Input((T, 1)), L.Dense(16, activation="relu"), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
p1, m1 = fit_eval([L.Input((T, 1)), L.SimpleRNN(16, return_sequences=True), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
acc_x_only = (p0 == yte).mean(); acc_rnn = (p1 == yte).mean()
print("x only:", round(acc_x_only, 3), " RNN:", round(acc_rnn, 3), " output shape:", m1.output_shape)
assert m1.output_shape == (None, 20, 1) and acc_rnn > acc_x_only + 0.1; print("OK")

In [ ]:
import tensorflow as tf
L = tf.keras.layers
T = 20

def make_cum(N, seed):
    r = np.random.RandomState(seed)
    x = r.randn(N, T, 1).astype("float32")
    y = (np.cumsum(x[:, :, 0], axis=1) > 0).astype("float32")
    return x, y

xtr, ytr = make_cum(3000, 0); xte, yte = make_cum(1000, 1)

def fit_eval(layers, xtr, ytr, xte, yte, seed=0):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential(layers)
    m.compile(tf.keras.optimizers.Adam(0.01), "binary_crossentropy")
    m.fit(xtr, ytr, epochs=15, batch_size=64, verbose=0)
    p = (m.predict(xte, verbose=0)[..., 0] > 0.5)
    return p, m

p0, _ = fit_eval([L.Input((T, 1)), L.Dense(16, activation="relu"), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
p1, m1 = fit_eval([L.Input((T, 1)), L.SimpleRNN(16, return_sequences=___), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
acc_x_only = (p0 == yte).mean(); acc_rnn = (p1 == yte).mean()
print("x only:", round(acc_x_only, 3), " RNN:", round(acc_rnn, 3), " output shape:", m1.output_shape)
assert m1.output_shape == (None, 20, ___) and acc_rnn > acc_x_only + 0.1; print("OK")

## Task 4: Does the label need the future? Forward vs bidirectional
New label: `1` if the **next** number is positive. The last position has no next number, so we do not score it.

In [ ]:
def make_next(N, seed):
    r = np.random.RandomState(seed)
    x = r.randn(N, T, 1).astype("float32")
    y = np.zeros((N, T), "float32"); y[:, :-1] = (x[:, 1:, 0] > 0)
    return x, y

xtr, ytr = make_next(3000, 0); xte, yte = make_next(1000, 1)

pf, _ = fit_eval([L.Input((T, 1)), L.SimpleRNN(16, return_sequences=True), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
pb, _ = fit_eval([L.Input((T, 1)), L.Bidirectional(L.SimpleRNN(16, return_sequences=True)), L.Dense(1, activation="sigmoid")], xtr, ytr, xte, yte)
acc_fwd = (pf[:, :-1] == yte[:, :-1]).mean(); acc_bi = (pb[:, :-1] == yte[:, :-1]).mean()
print("forward RNN:", round(acc_fwd, 3), " bidirectional:", round(acc_bi, 3))
assert acc_fwd < 0.6 and acc_bi > 0.9; print("OK")

**Question (2 sentences):** when can you NOT use a bidirectional RNN, even if it would be more accurate?

## Task 5: Your own project
In 3 to 4 sentences: which of the five RNN types does your project need (one-to-one, one-to-many, many-to-one, same-length many-to-many, length-changing many-to-many)? Do you need bidirectional? What simple baseline would you try first?

*Your answer:*